# 21회차 실습: 타이타닉 891명을 살피고 빈칸 채우기

위에서부터 한 칸씩 `Shift + Enter`로 실행하며 따라와요. 칸 위 **나와야 하는 것**과 내 결과가 같으면 맞게 따라온 거예요.

- 프로젝트: 20회차의 `~/dev/my-pandas` (오늘 터미널에서 `uv add seaborn matplotlib`을 더해요)
- 타이타닉 데이터는 인터넷에서 받아 와요. 연결된 상태로 실행해요.
- 이 노트북의 절 번호는 레슨의 절 번호와 같아요. 채우기는 앞 칸의 결과 위에 쌓이니 순서대로 실행해요. 꼬이면 1절부터 다시 실행해요.

## 1. 준비: 도구 불러오기와 데이터 받기

**나와야 하는 것:** `pandas: 3.1.1 numpy: 2.5.5 seaborn: 0.13.2` (숫자는 조금 달라도 괜찮아요). `ModuleNotFoundError: No module named 'seaborn'`이 나오면 터미널에서 `uv add seaborn matplotlib`을 하고 VS Code 위쪽 **Restart**로 커널을 다시 켜요.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
# seaborn: 파이썬 시각화 라이브러리

print("pandas:", pd.__version__, "numpy:", np.__version__, "seaborn:", sns.__version__)

**나와야 하는 것:** 0~4번 승객 5행, 15열(가운데는 `...`으로 접혀요). 첫 행은 `0  3  male  22.0`으로 시작하고 `deck`은 `NaN`이에요.

In [ ]:
df = sns.load_dataset('titanic')
df.head(5)

## 2. 첫인상: shape, info, describe, 빈칸 세기

**나와야 하는 것:** `(891, 15)`. `info()`에서 `age 714 non-null`, `embarked 889`, `deck 203`, `embark_town 889`이고 나머지는 891이에요.

In [ ]:
print(df.shape)
df.info()

**나와야 하는 것:** 숫자 열 6개의 요약표. `age`의 count가 714, mean이 29.699118, `fare`의 max가 512.329200이에요.

In [ ]:
df.describe()

**나와야 하는 것:** `age 177`, `embarked 2`, `deck 688`, `embark_town 2`, 나머지는 0

In [ ]:
df.isnull().sum()

**나와야 하는 것:** `나이 빈칸:  177 명`과 177행짜리 표(`age`가 모두 `NaN`). `empty_age`는 3절에서 다시 써요.

In [ ]:
# 나이가 비어 있는 승객 수, 성별, 클래스
empty_age = df['age'].isnull()
print('나이 빈칸: ', empty_age.sum(), '명')
df.loc[empty_age, ['sex', 'age', 'class']]

## 3. 숫자 빈칸: 평균으로 채우기

채우기 **전에** 평균과 중앙값을 봐요.

**나와야 하는 것:**

```text
평균: 30
중앙값: 28
```

In [ ]:
print("평균:", round(df['age'].mean()))
print("중앙값:", round(df['age'].median()))

**나와야 하는 것:** 아까 `NaN`이던 승객들(5, 17, 19, 26, 28번)의 `age`가 모두 `29.699118`

In [ ]:
mean_age = df['age'].mean()
# fillna : 빈칸(NaN)만 주어진 값으로 채운 새 Series 를 돌려준다. 다시 넣어야 표가 바뀐다
df.loc[:,'age'] = df['age'].fillna(mean_age)
df.loc[empty_age, ['sex', 'age', 'class']].head()

**주의:** 위 칸을 실행한 뒤 평균, 중앙값 칸을 다시 실행하면 채운 값이 섞여 중앙값이 30으로 나와요. 노트북은 실행한 순서대로 결과가 쌓여요.

## 4. 글자 빈칸: 최빈값 mode()[0]

**나와야 하는 것:** `Southampton 644`, `Cherbourg 168`, `Queenstown 77`, `NaN 2`

In [ ]:
# 탑승 항구 : 최빈값
df['embark_town'].value_counts(dropna=False)

### 일부러 틀린 코드 먼저

수업에서 처음 쓴 코드예요. 오류가 나지 않는데도 **빈칸이 그대로** 남아요.

**나와야 하는 것:** `최빈값:  0    Southampton`처럼 앞에 `0`이 붙고, 아래 표에 `NaN 2`가 그대로 있어요. `mode()`가 값 하나가 아니라 Series를 돌려줘서 짝을 못 찾았기 때문이에요.

In [ ]:
mode_town = df["embark_town"].mode()
print('최빈값: ', mode_town)

df.loc[:, "embark_town"] = df['embark_town'].fillna(mode_town)
df['embark_town'].value_counts(dropna=False)

### 고친 코드: [0]으로 값 하나를 꺼낸다

**나와야 하는 것:** `Southampton 646`, `Cherbourg 168`, `Queenstown 77`. `NaN` 줄이 사라져요.

In [ ]:
mode_town = df["embark_town"].mode()[0]          # 'Southampton'
df.loc[:, "embark_town"] = df['embark_town'].fillna(mode_town)
df['embark_town'].value_counts(dropna=False)

### embarked는 embark_town의 첫 글자로

**나와야 하는 것:** `embarked 0`, `embark_town 0`. 수업 노트북은 틀린 코드 상태에서 이 칸을 실행해 둘 다 2가 남았어요. 원본이 `NaN`이면 첫 글자도 `NaN`이라서, 바탕 열을 먼저 채워야 해요.

In [ ]:
# embarked town
df.loc[:, 'embarked'] = df['embark_town'].str[0]
df[['embarked', 'embark_town']].isnull().sum()

## 5. category 열: 'Unknown'부터 명단에 넣기

**나와야 하는 것:** `category`와 `Index(['A', 'B', 'C', 'D', 'E', 'F', 'G'], dtype='str')`

In [ ]:
print(df['deck'].dtype)
df['deck'].cat.categories

**해 보기:** 아래 칸의 `#`을 지우고 실행해 보세요. 명단에 없는 값을 넣으려 해서 `TypeError: Cannot setitem on a Categorical with a new category (Unknown), set the categories first`가 나와요. 확인했으면 다시 `#`을 붙여요.

In [ ]:
# df['deck'].fillna('Unknown')

**나와야 하는 것:** `['A', 'B', 'C', 'D', 'E', 'F', 'G', 'Unknown']`과, 가장 많은 `Unknown 688` 다음에 `C 59`, `B 47`... 순서의 개수

In [ ]:
df['deck'] = df['deck'].cat.add_categories('Unknown').fillna('Unknown')
print(df['deck'].cat.categories.to_list())
print(df['deck'].value_counts())

### 다시 확인

**나와야 하는 것:** 15열 모두 `0`. 하나라도 0이 아니면 앞 절의 채우기 칸을 순서대로 다시 실행해요.

In [ ]:
df.isnull().sum()

## 6. 튀는 값: IQR

**나와야 하는 것:** 두 번 모두 25% `7.9104`, 50% `14.4542`, 75% `31.0000`

In [ ]:
print(df['fare'].quantile([0.25,0.5,0.75]))
df['fare'].describe()[["25%", "50%", "75%"]]

**나와야 하는 것:**

```text
Q1 = 7.91, Q3 = 31.00 IQR = 23.09 UPPER = 65.63
이상 승선요금:  116 명
```

In [ ]:
# IQR 계산 (Q3 - Q1)
# 이상수치 Q3 + 1.5 * IQR

q1 = df['fare'].quantile(0.25)
q3 = df['fare'].quantile(0.75)
iqr = q3 - q1
upper = q3 + 1.5 * iqr

print(f"Q1 = {q1:.2f}, Q3 = {q3:.2f} IQR = {iqr:.2f} UPPER = {upper:.2f}")
print("이상 승선요금: ",(df["fare"] > upper).sum(), "명")

**나와야 하는 것:** 요금이 512.3292인 3행

In [ ]:
df[df['fare'] > 512]

**나와야 하는 것:** `891 => 888`. 이 칸은 한 번만 실행해요(다시 실행해도 888이지만, 위의 3행 칸은 이제 빈 표가 돼요).

In [ ]:
before = len(df)
df = df[df['fare'] < 512].copy()
print(before, "=>", len(df))

## 7. 중복 행: 세어 보기만

**나와야 하는 것:** `중복 행: 107 개`와 5행. 5행 모두 요금 `0.0`, 나이 `29.699118`인 2등석 남성으로 모든 열이 똑같아요(277, 413, 466, 481, 674번). 이름이 없는 표라서 다른 사람도 중복으로 잡혀요.

In [ ]:
print("중복 행:", df.duplicated().sum(), '개')
df[df.duplicated(keep=False)].sort_values(['fare', 'age']).head(5)

## 8. 히스토그램: 평균으로 채운 나이

**나와야 하는 것:** 파란 막대 20개, 제목 `나이분포`. 30세 바로 앞 막대 하나(28~32세, 261명)가 다른 막대보다 훨씬 높이 솟아요. 빈칸 177명이 모두 29.7세가 됐기 때문이에요. 한글이 네모로 나오면 글꼴 줄을 확인해요.

In [ ]:
import platform
import matplotlib.pyplot as plt

os_name = platform.system()
if os_name == "Windows":
    plt.rcParams['font.family'] = "Malgun Gothic"
elif os_name == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"

# 나이 분포 차트
fig, ax = plt.subplots(figsize=(6,4))
ax.hist(df["age"], bins = 20, color="blue", edgecolor="black")
ax.set_title("나이분포")
ax.set_xlabel("나이(세)")
plt.show()

## 혼자 치기

채우기 전 원본으로 연습해요. 먼저 아래 칸으로 원본을 새로 받아 `raw`에 담아요(위의 `df`는 이미 채워져 있어요). 정답은 맨 아래에 있어요.

In [ ]:
raw = sns.load_dataset('titanic')
raw.shape

**문제 1.** `raw`를 복사한 `r = raw.copy()`의 `age` 빈칸을 **중앙값**으로 채우고, 빈칸 수와 나이가 정확히 28인 승객 수를 찍어 보세요.

In [ ]:
# 여기에 써 보세요

**문제 2.** `raw['fare']`의 IQR 하한선 `Q1 - 1.5 × IQR`을 소수 둘째 자리까지 구하고, 그보다 작은 요금이 몇 명인지 세어 보세요.

In [ ]:
# 여기에 써 보세요

**문제 3.** `raw`에서 객실 등급(`class`)별 평균 나이를 소수 첫째 자리까지 구해 보세요. 20회차의 `groupby`를 써요(`observed=True`를 넣어요).

In [ ]:
# 여기에 써 보세요

**문제 4.** `raw`에서 성별(`sex`)별 생존율(`survived`의 평균)을 소수 둘째 자리까지 구해 보세요.

In [ ]:
# 여기에 써 보세요

**문제 5.** `raw['deck']`이 비어 있는 비율을 퍼센트로, 소수 첫째 자리까지 구해 보세요. `isnull()`의 평균이 비율이에요.

In [ ]:
# 여기에 써 보세요

## 정답

먼저 풀어 본 뒤에 봐요.

문제 1

```python
r = raw.copy()
r.loc[:, "age"] = r["age"].fillna(r["age"].median())
print(r["age"].isnull().sum(), (r["age"] == 28).sum())
# 0 202  (원래 28세 25명 + 채운 177명. 중앙값으로 채워도 막대 하나가 솟아요)
```

문제 2

```python
q1 = raw["fare"].quantile(0.25)
q3 = raw["fare"].quantile(0.75)
lower = q1 - 1.5 * (q3 - q1)
print(round(lower, 2), (raw["fare"] < lower).sum())
# -26.72 0  (하한선이 음수라 아래로 튀는 요금은 없어요)
```

문제 3

```python
raw.groupby("class", observed=True)["age"].mean().round(1)
# First 38.2, Second 29.9, Third 25.1
```

문제 4

```python
raw.groupby("sex")["survived"].mean().round(2)
# female 0.74, male 0.19
```

문제 5

```python
print(round(raw["deck"].isnull().mean() * 100, 1))
# 77.2
```